In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint

CSV_PATH = "findex_pakistan_with_clusters.csv"
data = pd.read_csv(CSV_PATH)
print(f"Loaded {data.shape[0]} rows, {data.shape[1]} columns")

Loaded 1000 rows, 25 columns


In [2]:
def wilson_ci(successes, n, label):
    """Wilson score 95% CI - preferred over normal approx for small n
    or proportions near 0/1."""
    low, high = proportion_confint(successes, n, alpha=0.05, method="wilson")
    pct = successes / n * 100
    print(f"  {label}: {pct:.1f}% (95% CI: {low*100:.1f}%-{high*100:.1f}%), n={n}")
    return low, high


def chi_square_test(data, col1, col2, label):
    """Chi-square test of independence + Cramer's V effect size."""
    contingency = pd.crosstab(data[col1], data[col2])
    chi2, p, dof, expected = chi2_contingency(contingency)
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan

    print(f"\n{label}")
    print(f"  Chi-square = {chi2:.3f}, p = {p:.4f}, dof = {dof}")
    print(f"  Cramer's V (effect size) = {cramers_v:.3f}")
    if (expected < 5).any():
        print("  WARNING: some expected cell counts < 5 - chi-square may be unreliable, "
              "consider Fisher's exact test for the affected cells")
    sig = "significant" if p < 0.05 else "NOT significant"
    print(f"  -> Association is {sig} at alpha=0.05")
    return chi2, p, cramers_v

In [3]:
# ---------------------------------------------------------------------
# 1. H1: confidence interval on the dormant-account rate
# ---------------------------------------------------------------------
print("=" * 70)
print("H1 - Access-Usage Gap: Confidence Intervals")
print("=" * 70)

account_holders = data[data["account"] == 1]
dormant = ((account_holders["dig_account"] == 0) & (account_holders["anydigpayment"] == 0)).sum()
wilson_ci(dormant, len(account_holders), "Dormant account rate (H1, original definition)")

# Cluster-based H1 confirmation: % of Cluster 1 still Excluded despite access
cluster1 = data[data["cluster"] == 1]
cluster1_excluded = (cluster1["adoption_tier"] == 0).sum()
wilson_ci(cluster1_excluded, len(cluster1),
          "Cluster 1 ('Connected but Unengaged') still formally Excluded")

H1 - Access-Usage Gap: Confidence Intervals
  Dormant account rate (H1, original definition): 6.6% (95% CI: 4.4%-9.9%), n=318
  Cluster 1 ('Connected but Unengaged') still formally Excluded: 66.8% (95% CI: 62.3%-71.0%), n=443


(0.623043032758913, 0.7104085686060128)

In [4]:
# ---------------------------------------------------------------------
# 2. H2: chi-square on cluster vs urban/rural, and tier vs urban/rural
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("H2 - Urban/Rural Association Tests")
print("=" * 70)

chi_square_test(data, "cluster", "is_rural", "Cluster vs Rural/Urban")
chi_square_test(data, "adoption_tier", "is_rural", "Adoption Tier vs Rural/Urban")


H2 - Urban/Rural Association Tests

Cluster vs Rural/Urban
  Chi-square = 19.823, p = 0.0000, dof = 2
  Cramer's V (effect size) = 0.141
  -> Association is significant at alpha=0.05

Adoption Tier vs Rural/Urban
  Chi-square = 21.742, p = 0.0001, dof = 3
  Cramer's V (effect size) = 0.147
  -> Association is significant at alpha=0.05


(np.float64(21.74199560030358),
 np.float64(7.381172057762268e-05),
 np.float64(0.14745167208378337))

In [5]:
# ---------------------------------------------------------------------
# 3. H3 / cluster structure: chi-square on cluster vs gender
#    (the PC1 loadings suggested gender + access are structurally linked)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("Cluster Structure - Gender Association")
print("=" * 70)

chi_square_test(data, "cluster", "female", "Cluster vs Gender")
chi_square_test(data, "adoption_tier", "female", "Adoption Tier vs Gender")


Cluster Structure - Gender Association

Cluster vs Gender
  Chi-square = 650.549, p = 0.0000, dof = 2
  Cramer's V (effect size) = 0.807
  -> Association is significant at alpha=0.05

Adoption Tier vs Gender
  Chi-square = 162.080, p = 0.0000, dof = 3
  Cramer's V (effect size) = 0.403
  -> Association is significant at alpha=0.05


(np.float64(162.08002702837763),
 np.float64(6.519724848927622e-35),
 np.float64(0.4025916380507395))

In [6]:
# ---------------------------------------------------------------------
# 4. Confidence intervals on key cluster profile percentages
#    (the numbers likely to be quoted directly in the write-up)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("Cluster Profile - Confidence Intervals on Headline Numbers")
print("=" * 70)

for c in sorted(data["cluster"].unique()):
    sub = data[data["cluster"] == c]
    n = len(sub)
    print(f"\nCluster {c} (n={n}, {n/len(data)*100:.1f}% of sample):")
    wilson_ci(int((sub["female"] == 1).sum()), n, "  % female")
    wilson_ci(int((sub["is_rural"] == 1).sum()), n, "  % rural")
    wilson_ci(int((sub["con1"] == 1).sum()), n, "  % mobile phone")
    wilson_ci(int((sub["internet_use"] == 1).sum()), n, "  % internet use")
    wilson_ci(int((sub["adoption_tier"] == 0).sum()), n, "  % Excluded")


Cluster Profile - Confidence Intervals on Headline Numbers

Cluster 0 (n=431, 43.1% of sample):
    % female: 96.3% (95% CI: 94.1%-97.7%), n=431
    % rural: 51.0% (95% CI: 46.3%-55.7%), n=431
    % mobile phone: 28.3% (95% CI: 24.3%-32.7%), n=431
    % internet use: 9.7% (95% CI: 7.3%-12.9%), n=431
    % Excluded: 89.3% (95% CI: 86.1%-91.9%), n=431

Cluster 1 (n=443, 44.3% of sample):
    % female: 16.3% (95% CI: 13.1%-20.0%), n=443
    % rural: 46.0% (95% CI: 41.5%-50.7%), n=443
    % mobile phone: 98.0% (95% CI: 96.2%-98.9%), n=443
    % internet use: 62.3% (95% CI: 57.7%-66.7%), n=443
    % Excluded: 66.8% (95% CI: 62.3%-71.0%), n=443

Cluster 2 (n=126, 12.6% of sample):
    % female: 10.3% (95% CI: 6.1%-16.9%), n=126
    % rural: 28.6% (95% CI: 21.4%-37.0%), n=126
    % mobile phone: 98.4% (95% CI: 94.4%-99.6%), n=126
    % internet use: 93.7% (95% CI: 88.0%-96.7%), n=126
    % Excluded: 0.8% (95% CI: 0.1%-4.4%), n=126


In [7]:
# ---------------------------------------------------------------------
# 5. Model performance: bootstrap CI on best model's macro F1
#    (quick check that 0.486 macro F1 isn't a fluke of one split)
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("Model Robustness - Bootstrap CI on Macro F1 (Random Forest)")
print("=" * 70)
print("NOTE: this requires re-running the Week 2 train/test split with")
print("multiple random seeds. See 04b_bootstrap_f1.py if you want this")
print("as a standalone check - flag if useful and it can be added.")


Model Robustness - Bootstrap CI on Macro F1 (Random Forest)
NOTE: this requires re-running the Week 2 train/test split with
multiple random seeds. See 04b_bootstrap_f1.py if you want this
as a standalone check - flag if useful and it can be added.


In [8]:
# ---------------------------------------------------------------------
# 6. Summary table of all headline numbers for easy copy into write-up
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("SUMMARY - Numbers Ready to Quote (with 95% CI where applicable)")
print("=" * 70)
print("""
Copy the printed values above into your results section. Suggested
structure per hypothesis:

H1: "X% of account holders (95% CI: Y-Z%) show no digital usage"
    "In the independent cluster analysis, X% of Cluster 1 (95% CI:
    Y-Z%) remain formally Excluded despite near-universal access"

H2: "Chi-square test confirms adoption tier is significantly
    associated with urban/rural status (chi2=X, p=Y, Cramer's V=Z)"

H3: "SHAP analysis and chi-square testing both confirm gender is
    significantly associated with cluster membership (chi2=X, p=Y),
    exceeding the effect size of rural/urban status"
""")


SUMMARY - Numbers Ready to Quote (with 95% CI where applicable)

Copy the printed values above into your results section. Suggested
structure per hypothesis:

H1: "X% of account holders (95% CI: Y-Z%) show no digital usage"
    "In the independent cluster analysis, X% of Cluster 1 (95% CI:
    Y-Z%) remain formally Excluded despite near-universal access"

H2: "Chi-square test confirms adoption tier is significantly
    associated with urban/rural status (chi2=X, p=Y, Cramer's V=Z)"

H3: "SHAP analysis and chi-square testing both confirm gender is
    significantly associated with cluster membership (chi2=X, p=Y),
    exceeding the effect size of rural/urban status"

